# Préparation des données : de l’exploration au corpus d’entraînement

Le projet demande d’adapter un modèle au **triage médical**, mais les quatre sources disponibles n’ont ni le même format ni le même objectif.

Je cherche donc à répondre progressivement à trois questions :

1. Que contient réellement chaque source ?
2. Pour quelle tâche chaque contenu peut-il être utile ?
3. Comment construire une seconde phase de SFT centrée sur le triage ?


In [ ]:
import csv
import hashlib
import heapq
import json
import re
import runpy
import sys
from collections import Counter
from pathlib import Path

import ijson
import pyarrow.parquet as pq

RACINE = Path.cwd().resolve()
if RACINE.name == "data_ingestion":
    RACINE = RACINE.parent
sys.path.insert(0, str(RACINE / "data_ingestion"))
BRUT = RACINE / "data/raw"
ANNOTATIONS = RACINE / "data/annotations"
SORTIE = RACINE / "data/processed/sft_deux_phases"

def lire_jsonl(chemin):
    """Charge un petit fichier JSONL en mémoire pour l’exploration."""
    with chemin.open(encoding="utf-8-sig") as fichier:
        return [json.loads(ligne) for ligne in fichier if ligne.strip()]


def executer_script(nom, *arguments):
    """Exécute un script local comme en ligne de commande, puis restaure sys.argv."""
    ancien_argv = sys.argv[:]
    try:
        chemin = RACINE / "data_ingestion" / nom
        sys.argv = [str(chemin), *arguments]
        return runpy.run_path(str(chemin), run_name="__main__")
    finally:
        sys.argv = ancien_argv


## 1. Inventaire initial

À ce stade, je connais seulement la langue, la licence déclarée et l’emplacement des fichiers. Je ne fixe pas encore leur usage.

| Source | Langue dominante | Licence constatée |
|---|---:|---|
| MediQAl | français | CC BY 4.0 |
| FrenchMedMCQA | français | non renseignée |
| MedQuAD | anglais | non renseignée |
| UltraMedical-Preference | anglais | MIT |

Une compatibilité technique ne suffit pas : une licence absente limite la réutilisation et la diffusion du corpus dérivé.


## 2. Exploration des formats


### 2.1 MediQAl


In [ ]:
# Je mesure les volumes et la présence de cas cliniques.
for configuration in ("mcqu", "mcqm", "oeq"):
    for split in ("train", "validation", "test"):
        chemin = BRUT / "MediQAl" / configuration / f"{split}.json"
        if not chemin.exists():
            continue
        lignes = lire_jsonl(chemin)
        avec_cas = sum(bool((x.get("clinical_case") or "").strip()) for x in lignes)
        print(configuration, split, {
            "lignes": len(lignes),
            "avec_cas_clinique": avec_cas,
            "champs": list(lignes[0]),
        })


**Constat.** MediQAl mélange des questions corrigées et des présentations cliniques. Les premières peuvent fournir une réponse cible ; les secondes ressemblent au contexte du triage, mais ne contiennent pas de niveau ESI de référence.


### 2.2 FrenchMedMCQA


In [ ]:
# Je vérifie le schéma des trois splits Parquet.
for chemin in sorted((BRUT / "frenchmedmcqa/data").glob("*.parquet")):
    table = pq.read_table(chemin)
    print(chemin.name, {"lignes": table.num_rows, "champs": table.column_names})


**Constat.** La source contient des QCM français avec leur bonne réponse. Elle est compatible avec une tâche de QA, mais elle n’apporte ni réponse rédigée ni présentation destinée au triage.


### 2.3 MedQuAD


In [ ]:
# Je compte les couples question-réponse et leurs catégories.
chemin = BRUT / "MedQuad-MediacalQnADataset/medDataset_processed.csv"
with chemin.open(encoding="utf-8-sig", newline="") as fichier:
    lignes = list(csv.DictReader(fichier))
print({
    "lignes": len(lignes),
    "champs": list(lignes[0]),
    "qtypes_principaux": Counter(x["qtype"] for x in lignes).most_common(8),
})


**Constat.** MedQuAD fournit directement des réponses médicales rédigées en anglais. Le format est adapté à la QA conversationnelle. L’absence de licence renseignée reste toutefois un point à résoudre avant toute diffusion.


### 2.4 UltraMedical-Preference


In [ ]:
# Le fichier est volumineux : je lis un exemple en flux sans tout charger en mémoire.
chemin = BRUT / "UltraMedical-Preference/data/train.json"
with chemin.open("rb") as fichier:
    premier = next(ijson.items(fichier, "item"))
print({cle: type(valeur).__name__ for cle, valeur in premier.items()})
print("Sous-source :", premier.get("prompt_id", "").split(",", 1)[0])


**Constat.** UltraMedical associe une sollicitation à des réponses préférées et rejetées. Ce schéma correspond naturellement au DPO. Certaines sollicitations décrivent aussi un patient : elles peuvent devenir candidates au triage après filtrage et annotation.


## 3. Déduire les usages à partir des observations

J’évalue chaque source avec quatre critères :

```text
contenu disponible
        │
        ├── correspond-il à la tâche ?
        ├── existe-t-il une cible fiable ?
        ├── faut-il créer une nouvelle annotation ?
        └── la licence autorise-t-elle l’usage envisagé ?
```

| Source | Usage techniquement possible | Usage retenu | Motif principal |
|---|---|---|---|
| MediQAl | QA, QCM, cas cliniques | QA + candidats au triage | réponses corrigées et cas français sous CC BY 4.0 |
| FrenchMedMCQA | QA/QCM | exploration | redondant avec MediQAl et licence absente |
| MedQuAD | QA rédigée | QA anglaise expérimentale | réponses riches, sous réserve de clarification de la licence |
| UltraMedical | QA, préférences, cas cliniques | candidats au triage + DPO | préférences natives et licence MIT |

Le **triage** est donc un usage dérivé : aucune source ne fournit directement les niveaux ESI nécessaires.


## 4. Construire une cible de triage fiable

Une présentation clinique ne devient pas automatiquement un exemple d’entraînement. Je dois d’abord vérifier qu’elle décrit l’état du patient au premier accueil, puis produire une cible ESI.

```text
cas clinique brut
      │
      ▼
qualification temporelle ── non exploitable ─► exclusion
      │ exploitable
      ▼
annotation indépendante par plusieurs modèles
      │
      ▼
majorité qualification → majorité urgence → majorité ESI exact
      │                                      │
      └──────────── désaccord ────────────────┴─► non retenu
```

Le consensus réduit la dépendance à un annotateur unique. Il reste une méthode de constitution de corpus et non une validation clinique.


### Scripts communs à l’annotation

- [`schema_annotation.py`](schema_annotation.py) définit le schéma attendu et vérifie les citations, les valeurs ESI et les abstentions.
- La consigne versionnée se trouve dans [`consigne_annotation.txt`](consigne_annotation.txt).

Ces deux fichiers assurent que tous les annotateurs sont contrôlés avec les mêmes règles.


In [ ]:
NIVEAU_URGENCE = {
    1: "maximale", 2: "maximale", 3: "moderee", 4: "differee", 5: "differee"
}

def valider_classement(annotation):
    """Vérifie la cohérence minimale entre le statut, l’ESI et l’urgence."""
    if annotation["statut_classement"] == "classe":
        esi = annotation["esi"]
        if type(esi) is not int or NIVEAU_URGENCE.get(esi) != annotation["urgence"]:
            raise ValueError("Classement ESI incohérent")
    elif annotation["esi"] is not None or annotation["urgence"] is not None:
        raise ValueError("Une abstention doit avoir des valeurs nulles")
    return annotation

# schema_annotation.py ajoute les contrôles complets : citations exactes,
# temporalité, qualification et cohérence entre tous les champs.


### 4.1 Première source de triage : MediQAl


**Chaîne de production MediQAl**

```text
preparer_corpus.py
        ↓ 2 007 cas répartis en lots
annotations Codex + Claude
        ↓
lancer_batch_mistral.py
        ↓ troisième vote
consolider_consensus_trois.py
        ↓
consensus_3_modeles.jsonl
```

- [`preparer_corpus.py`](preparer_corpus.py) déduplique les cas et construit les lots reproductibles.
- [`lancer_batch_mistral.py`](lancer_batch_mistral.py) prépare, suit et récupère le troisième vote.
- [`consolider_consensus_trois.py`](consolider_consensus_trois.py) valide les trois sorties et calcule la majorité.


In [ ]:
# Étape locale : je reconstruis les cas, les lots et leur manifeste.
executer_script("preparer_corpus.py")


In [ ]:
# Étapes externes : elles restent volontairement commentées.
# annoter_avec_api_codex("data/annotations/corpus/entrees/lots")
# annoter_avec_api_claude("data/annotations/corpus/entrees/lots")
# executer_script("lancer_batch_mistral.py", "--submit")
# executer_script("lancer_batch_mistral.py", "--status")
# executer_script("lancer_batch_mistral.py", "--download")

print("Aucun appel externe lancé. Je réutilise les annotations déjà présentes.")


In [ ]:
def extraire_cas_mediqal():
    """Réunit les cas MCQU et MCQM en dédupliquant leur texte normalisé."""
    cas_par_texte = {}
    for configuration in ("mcqu", "mcqm"):
        chemin = BRUT / "MediQAl" / configuration / "train.json"
        for entree in lire_jsonl(chemin):
            texte = (entree.get("clinical_case") or "").strip()
            cle = " ".join(texte.split())
            if not cle:
                continue
            cas = cas_par_texte.setdefault(cle, {
                "case_id": "mediqal-" + hashlib.sha256(cle.encode()).hexdigest(),
                "texte_source": texte,
                "references": [],
            })
            cas["references"].append({
                "configuration": configuration,
                "question_id": entree["id"],
            })
    return sorted(cas_par_texte.values(), key=lambda x: x["case_id"])

cas_mediqal = extraire_cas_mediqal()
assert len(cas_mediqal) == 2007
len(cas_mediqal)


In [ ]:
# Je lis les résultats produits après annotation des 2 007 cas distincts.
rapport_mediqal = json.loads(
    (ANNOTATIONS / "corpus/rapport_consensus_3_modeles.json").read_text()
)
{
    "cas_analyses": rapport_mediqal["nombre_cas"],
    "statuts": rapport_mediqal["statuts"],
    "esi_retenus": rapport_mediqal["esi_majoritaires_retenus"],
}


**Résultat.** Le consensus retient 754 cas sur l’urgence. Parmi eux, 701 possèdent aussi une majorité sur le niveau ESI exact. La déduplication du message utilisateur ramène ce jeu à **700 exemples uniques**.

**Enseignement.** Une majorité sur une catégorie large comme l’urgence ne suffit pas pour entraîner une sortie ESI précise : le niveau exact doit lui aussi faire consensus.


In [ ]:
def majorite(valeurs):
    """Retourne une valeur seulement si au moins deux annotateurs sont d’accord."""
    valeur, votes = Counter(valeurs).most_common(1)[0]
    return valeur if votes >= 2 else None

def consensus_esi(annotations):
    """Calcule le consensus hiérarchique utilisé pour sélectionner un cas."""
    decision = majorite([x["decision_entree"] for x in annotations])
    if decision != "candidat_initial":
        return None
    classes = [x for x in annotations if x["statut_classement"] == "classe"]
    urgence = majorite([x["urgence"] for x in classes]) if classes else None
    esi = majorite([x["esi"] for x in classes if x["urgence"] == urgence]) if urgence else None
    return {"urgence": urgence, "esi": esi} if esi is not None else None


In [ ]:
# Étape locale : les sorties existantes sont validées et consolidées.
executer_script("consolider_consensus_trois.py")


### 4.2 Seconde source de triage : UltraMedical


**Chaîne de préparation UltraMedical**

- [`extraire_candidats_ultramedical.py`](extraire_candidats_ultramedical.py) filtre les présentations de patient et applique les quotas par sous-source.
- [`preparer_corpus_ultramedical.py`](preparer_corpus_ultramedical.py) transforme les 2 000 candidats en 40 lots d’annotation.
- [`consolider_ultramedical.py`](consolider_ultramedical.py) valide les sorties et exige une majorité sur la qualification, l’urgence et l’ESI exact.

Je conserve le volume réellement soutenu par le consensus. Je ne relâche pas la règle pour atteindre un quota fixé à l’avance.


In [ ]:
# Étapes locales : sélection en flux, manifeste et lots d'annotation.
executer_script("extraire_candidats_ultramedical.py")
executer_script("preparer_corpus_ultramedical.py")


In [ ]:
# Étapes externes : à lancer seulement dans les sessions prévues.
# annoter_ultramedical_avec_api_codex(...)
# annoter_ultramedical_avec_api_claude(...)
# executer_script("lancer_batch_ultramedical_mistral.py", "--submit")
# executer_script("lancer_batch_ultramedical_mistral.py", "--status")
# executer_script("lancer_batch_ultramedical_mistral.py", "--download")

print("Aucun appel externe lancé. La suite reprendra quand les sorties seront complètes.")


### Reprise locale après les annotations UltraMedical

La cellule suivante exploite les 40 lots Codex, les 40 lots Claude et les 1 516 votes Mistral ciblés. Elle ne contacte aucun service. Le consensus exact retient 489 cas sur 2 000 ; les autres sont exclus ou conservés pour revue.


In [ ]:
BASE_ULTRA = ANNOTATIONS / "ultramedical_triage"

executer_script("consolider_ultramedical.py")
selection_ultra = lire_jsonl(BASE_ULTRA / "consensus_retenus.jsonl")
rapport_ultra = json.loads((BASE_ULTRA / "rapport_consensus.json").read_text())
{
    "cas_sources": rapport_ultra["cas_sources"],
    "consensus_exacts": rapport_ultra["consensus_exacts"],
    "ecart_objectif_initial": rapport_ultra["ecart_objectif"],
    "esi": rapport_ultra["esi_retenus"],
}


In [ ]:
PRESENTATION_PATIENT = re.compile(
    r"\b(I have|I am|my |patient|woman|man|boy|girl|infant|child|year[- ]old|presents?|reports?)\b",
    re.IGNORECASE,
)
OPTIONS_QCM = re.compile(r"(?:^|\n)\s*[A-E][.)]\s+", re.MULTILINE)

def candidat_ultramedical(prompt):
    """Conserve une présentation assez détaillée et écarte les QCM explicites."""
    texte = " ".join((prompt or "").split())
    return 100 <= len(texte) <= 3000 and bool(PRESENTATION_PATIENT.search(texte)) \
        and not OPTIONS_QCM.search(prompt)

def ordre_reproductible(identifiant, seed=3407):
    """Produit une clé stable pour sélectionner un échantillon sans biais d’ordre."""
    return hashlib.sha256(f"{seed}:{identifiant}".encode()).hexdigest()

# Le parcours complet est effectué en flux dans extraire_candidats_ultramedical.py.


In [ ]:
# Je suis séparément la production de chaque annotateur.
lots_codex = len(list((ANNOTATIONS / "ultramedical_triage/codex").glob("lot-*.jsonl")))
lots_claude = len(list((ANNOTATIONS / "ultramedical_triage/claude").glob("lot-*.jsonl")))
{"lots_codex": f"{lots_codex}/40", "lots_claude": f"{lots_claude}/40"}


Je sélectionne 2 000 présentations anglaises candidates avant annotation. Codex et Claude ont chacun produit 40 lots complets et valides. Ils ne sont d’accord sur la qualification que pour **878 cas sur 2 000** ; parmi les 457 cas classés par les deux, ils partagent le même ESI pour **261 cas**. Ce désaccord justifie le troisième vote Mistral. L’objectif initial de 700 cas n’est pas un quota à remplir : seuls les 489 cas soutenus par le consensus exact seront retenus puis traduits en français.


## 5. Format commun nécessaire au SFT

Les sources sont conservées dans leur format brut. Je les convertis seulement à la fin vers un schéma commun :

```json
{
  "id": "source-identifiant",
  "messages": [
    {"role": "system", "content": "consigne stable"},
    {"role": "user", "content": "question ou présentation clinique"},
    {"role": "assistant", "content": "réponse cible"}
  ],
  "langue": "fr",
  "type_tache": "triage",
  "source": "...",
  "licence": "...",
  "niveau_confiance": "..."
}
```

Cette normalisation permet au même modèle d’apprendre plusieurs tâches tout en gardant la provenance hors de la conversation.


Le module [`outils_sft.py`](outils_sft.py) centralise cette conversion : lecture des sources, création des messages, déduplication, ordre reproductible et statistiques. Centraliser ces règles évite que deux scripts produisent des formats légèrement différents.


In [ ]:
SYSTEME_FR = (
    "Tu es un assistant médical bilingue destiné à un prototype pédagogique. "
    "Réponds directement et ne remplace jamais l’évaluation d’un professionnel de santé."
)
SYSTEME_EN = (
    "You are a bilingual medical assistant for an educational prototype. "
    "Answer directly and never replace assessment by a healthcare professional."
)

def exemple_sft(identifiant, utilisateur, assistant, **metadonnees):
    """Convertit une paire source en conversation tout en gardant sa provenance."""
    systeme = SYSTEME_EN if metadonnees.get("langue") == "en" else SYSTEME_FR
    return {
        "id": identifiant,
        "messages": [
            {"role": "system", "content": systeme},
            {"role": "user", "content": utilisateur.strip()},
            {"role": "assistant", "content": assistant.strip()},
        ],
        **metadonnees,
    }

def dedupliquer(exemples):
    """Conserve un seul exemple par message utilisateur normalisé."""
    sortie, vus = [], set()
    for exemple in exemples:
        cle = " ".join(exemple["messages"][1]["content"].split()).casefold()
        if cle and cle not in vus:
            vus.add(cle)
            sortie.append(exemple)
    return sortie


## 6. Proposition finale pour le SFT

L’exploration fait apparaître deux apprentissages complémentaires. Je propose de les appliquer successivement au **même adaptateur LoRA**.

```text
modèle de base
      │
      ▼
phase 1 — QA médicale bilingue
      │  apprendre à répondre directement et conserver les connaissances générales
      ▼
phase 2 — triage bilingue
         apprendre le format ESI, la priorité et l’orientation prudente
```

| Phase | Français | Anglais | Total | Fonction |
|---|---:|---:|---:|---|
| 1 — QA | 1 311 | 1 311 | 2 622 | socle médical général |
| 2 — triage | 1 189 | 1 189 | 2 378 | spécialisation au triage |
| **Ensemble** | **2 500** | **2 500** | **5 000** | corpus bilingue équilibré |

Le triage représente 47,6 % du total. Les deux étapes restent équilibrées tout en conservant leurs objectifs distincts.


### Composition obtenue pour la phase 2

```text
MediQAl :     700 cas FR ── traduction ──► 700 cas EN
UltraMedical : 489 cas EN ── traduction ──► 489 cas FR
                                         │
                                         ▼
                          2 378 exemples de triage
                          1 189 FR + 1 189 EN
```

Le volume UltraMedical résulte du consensus exact. Les deux versions linguistiques d’un même cas restent dans le même split afin d’éviter une fuite entre entraînement et validation.


**Scripts de construction SFT**

- [`preparer_sft_deux_phases.py`](preparer_sft_deux_phases.py) construit la phase QA, le triage MediQAl français et les splits reproductibles.
- [`traduire_triage_mistral.py`](traduire_triage_mistral.py) valide la traduction anglaise des 700 cas MediQAl et reconstitue les paires.
- [`preparer_sft_ultramedical.py`](preparer_sft_ultramedical.py) transforme les consensus UltraMedical exacts en exemples anglais.
- [`traduire_ultramedical_mistral.py`](traduire_ultramedical_mistral.py) prépare et récupère leur traduction française.

Ces scripts écrivent dans `data/processed/sft_deux_phases/`.


In [ ]:
# Étapes locales : je reconstruis les sorties disponibles dans data/processed.
executer_script("preparer_sft_deux_phases.py")
executer_script("traduire_triage_mistral.py")  # prépare seulement les requêtes locales

# Appels externes de traduction, volontairement désactivés :
# executer_script("traduire_triage_mistral.py", "--submit")
# executer_script("traduire_triage_mistral.py", "--status")
# executer_script("traduire_triage_mistral.py", "--download")


In [ ]:
def repartir_par_paire(exemples, proportion_validation=0.10, seed=3407):
    """Place toutes les traductions d’un même cas dans le même split."""
    groupes = {}
    for exemple in exemples:
        identifiant_paire = exemple.get("id_paire", exemple["id"])
        groupes.setdefault(identifiant_paire, []).append(exemple)
    identifiants = sorted(
        groupes,
        key=lambda x: hashlib.sha256(f"{seed}:{x}".encode()).hexdigest(),
    )
    limite = round(len(identifiants) * proportion_validation)
    validation = {x for x in identifiants[:limite]}
    train = [e for i, groupe in groupes.items() if i not in validation for e in groupe]
    val = [e for i, groupe in groupes.items() if i in validation for e in groupe]
    return train, val

# L’écriture finale est déclenchée dans la cellule suivante.


In [ ]:
# État des fichiers déjà préparés ; ce rapport ne vaut pas validation finale.
rapport = json.loads((SORTIE / "rapport.json").read_text())
{
    "strategie": rapport["strategie"],
    "qa_train": rapport["phase1_qa"]["train"]["nombre"],
    "qa_validation": rapport["phase1_qa"]["validation"]["nombre"],
    "triage_fr_disponible": sum(
        rapport["phase2_triage_disponible_fr"][split]["nombre"]
        for split in ("train", "validation")
    ),
    "total_vise": rapport["total_attendu"],
}


### Deux corpus, puis deux découpages indépendants

Je ne mélange pas les objectifs pédagogiques :

```text
phase 1 : question-réponse médicale générale ─► spécialisation au domaine médical
phase 2 : cas cliniques avec consensus ESI ───► spécialisation au triage
```

Chaque corpus maître est ensuite séparé en 80 % entraînement, 10 % validation et 10 % test. Pour le triage, je découpe les cas avant leurs traductions : les versions française et anglaise restent toujours dans le même jeu.


In [ ]:
executer_script("preparer_sft_ultramedical.py")
ultra_en = lire_jsonl(SORTIE / "phase2_ultramedical_en.jsonl")
print("Exemples UltraMedical anglais :", len(ultra_en))

# Appels externes volontairement désactivés dans le notebook :
# executer_script("traduire_ultramedical_mistral.py", "--submit")
# executer_script("traduire_ultramedical_mistral.py", "--status")
# executer_script("traduire_ultramedical_mistral.py", "--download")


In [ ]:
executer_script("assembler_phase2.py")
executer_script("repartir_corpus_sft.py")
rapport_splits = json.loads((SORTIE / "rapport_splits_sft.json").read_text())
rapport_splits


## 7. Contrôles avant entraînement

| Contrôle | Risque évité |
|---|---|
| identifiants uniques et JSONL valide | exemples perdus ou dupliqués |
| rôles `system/user/assistant` complets | format incompatible avec le chat template |
| déduplication du message utilisateur | surapprentissage d’un même cas |
| paire bilingue dans un seul split | fuite entre train, validation et test |
| jeu de test jamais utilisé pendant le SFT | évaluation biaisée |
| conservation de la provenance et de la licence | perte de traçabilité |
| exclusion des `prompt_id` SFT du DPO | réutilisation de la même sollicitation |
| audit des données personnelles | conservation de noms ou coordonnées plausibles |
| revue manuelle d’un échantillon | erreurs que les contrôles automatiques ne voient pas |

UltraMedical demande une attention particulière : certains textes contiennent des noms, des établissements ou des lieux ressemblant à des données réelles.


Le script [`auditer_anonymisation.py`](auditer_anonymisation.py) recherche les identifiants potentiels dans les fichiers JSONL produits. Son rapport guide la revue humaine ; il ne remplace pas cette revue.


In [ ]:
def verifier_exemples(exemples):
    """Contrôle le schéma conversationnel et l’unicité des identifiants."""
    ids = [x["id"] for x in exemples]
    if len(ids) != len(set(ids)):
        raise ValueError("Identifiants dupliqués")
    for exemple in exemples:
        roles = [message["role"] for message in exemple["messages"]]
        if roles != ["system", "user", "assistant"]:
            raise ValueError(f"Rôles incorrects : {exemple['id']}")
        if any(not message["content"].strip() for message in exemple["messages"]):
            raise ValueError(f"Message vide : {exemple['id']}")
    return len(exemples)


In [ ]:
# Audit local des fichiers actuellement disponibles.
executer_script("auditer_anonymisation.py")


## 8. Place du DPO

Le DPO intervient après le SFT. J’utilise les couples `chosen/rejected` natifs d’UltraMedical et j’exclus tous les `prompt_id` déjà sélectionnés pour le triage.

```text
UltraMedical train
    ├── prompts cliniques sélectionnés ─► SFT triage
    └── autres couples chosen/rejected ─► DPO
```

Cette séparation permet au DPO d’apprendre une préférence de réponse sans revoir les mêmes sollicitations que le SFT.


La cellule suivante construit directement le DPO. Le volume de 5 000 paires est un paramètre de travail adapté au projet ; il pourra être ajusté avant l’entraînement selon le budget de calcul. La sélection est déterministe et n’utilise aucune API.


In [ ]:
N_DPO = 5000
N_DPO_VALIDATION = 500

def extraire_reponse(messages):
    """Retourne la dernière réponse assistant non vide d’une conversation."""
    for message in reversed(messages or []):
        if message.get("role") == "assistant" and (message.get("content") or "").strip():
            return message["content"].strip()
    return None

def construire_dpo(nombre=N_DPO):
    """Sélectionne en flux des préférences disjointes du SFT et écrit deux splits."""
    base_ultra = ANNOTATIONS / "ultramedical_triage"
    cas_sft = lire_jsonl(base_ultra / "entrees/cas.jsonl")
    exclus = {ref["prompt_id"] for cas in cas_sft for ref in cas["references"]}
    tas, compteur, vus = [], 0, set()
    source = BRUT / "UltraMedical-Preference/data/train.json"
    with source.open("rb") as fichier:
        for entree in ijson.items(fichier, "item"):
            prompt_id = entree.get("prompt_id")
            prompt = (entree.get("prompt") or "").strip()
            choisie = extraire_reponse(entree.get("chosen"))
            rejetee = extraire_reponse(entree.get("rejected"))
            cle_prompt = " ".join(prompt.split()).casefold()
            if (not prompt_id or prompt_id in exclus or cle_prompt in vus
                    or not prompt or not choisie or not rejetee or choisie == rejetee):
                continue
            vus.add(cle_prompt)
            ordre = int(hashlib.sha256(f"3407:{prompt_id}".encode()).hexdigest(), 16)
            ligne = {
                "id": "ultramedical-dpo-" + hashlib.sha256(prompt_id.encode()).hexdigest(),
                "prompt": [{"role": "user", "content": prompt}],
                "chosen": [{"role": "assistant", "content": choisie}],
                "rejected": [{"role": "assistant", "content": rejetee}],
                "source": "TsinghuaC3I/UltraMedical-Preference",
                "source_split": "train", "prompt_id": prompt_id,
                "licence": "MIT", "label_type": entree.get("label_type"),
            }
            element = (-ordre, compteur, ligne)
            compteur += 1
            if len(tas) < nombre:
                heapq.heappush(tas, element)
            elif ordre < -tas[0][0]:
                heapq.heapreplace(tas, element)
    if len(tas) != nombre:
        raise ValueError(f"{nombre} préférences attendues, {len(tas)} obtenues")
    selection = [x[2] for x in sorted(tas, key=lambda x: -x[0])]
    validation, train = selection[:N_DPO_VALIDATION], selection[N_DPO_VALIDATION:]
    for nom, lignes in (("dpo_train.jsonl", train), ("dpo_validation.jsonl", validation)):
        (SORTIE / nom).write_text(
            "".join(json.dumps(x, ensure_ascii=False) + "\n" for x in lignes)
        )
    rapport_dpo = {
        "source": "TsinghuaC3I/UltraMedical-Preference", "licence": "MIT",
        "seed": 3407, "prompt_ids_exclus_du_sft": len(exclus),
        "train": len(train), "validation": len(validation),
        "selection": "5 000 plus petites empreintes après filtrage et déduplication",
    }
    (SORTIE / "rapport_dpo.json").write_text(
        json.dumps(rapport_dpo, ensure_ascii=False, indent=2) + "\n"
    )
    print("DPO :", len(train), "train +", len(validation), "validation")
    return train, validation

dpo_train, dpo_validation = construire_dpo()


## 9. Traçabilité des transformations

| Étape | Script principal | Sortie utile |
|---|---|---|
| lots MediQAl | [`preparer_corpus.py`](preparer_corpus.py) | `data/annotations/corpus/entrees/` |
| consensus MediQAl | [`consolider_consensus_trois.py`](consolider_consensus_trois.py) | `consensus_3_modeles.jsonl` |
| candidats UltraMedical | [`extraire_candidats_ultramedical.py`](extraire_candidats_ultramedical.py) | `ultramedical_triage/entrees/cas.jsonl` |
| lots UltraMedical | [`preparer_corpus_ultramedical.py`](preparer_corpus_ultramedical.py) | `ultramedical_triage/entrees/lots/` |
| consensus UltraMedical | [`consolider_ultramedical.py`](consolider_ultramedical.py) | `consensus_retenus.jsonl` |
| SFT disponible | [`preparer_sft_deux_phases.py`](preparer_sft_deux_phases.py) | `data/processed/sft_deux_phases/` |
| traduction MediQAl | [`traduire_triage_mistral.py`](traduire_triage_mistral.py) | `phase2_triage_{train,validation}.jsonl` |
| SFT UltraMedical | [`preparer_sft_ultramedical.py`](preparer_sft_ultramedical.py) | `phase2_ultramedical_en.jsonl` |
| traduction UltraMedical | [`traduire_ultramedical_mistral.py`](traduire_ultramedical_mistral.py) | `traduction_ultra_sorties.jsonl` |
| assemblage phase 2 | [`assembler_phase2.py`](assembler_phase2.py) | `phase2_triage_corpus.jsonl` |
| splits des deux phases | [`repartir_corpus_sft.py`](repartir_corpus_sft.py) | `{phase1_qa,phase2_triage}_{train,validation,test}.jsonl` |
| audit des identifiants | [`auditer_anonymisation.py`](auditer_anonymisation.py) | rapports d’audit dans `data/processed/` |
| DPO | cellule `construire_dpo` | `dpo_{train,validation}.jsonl` |

```text
data/raw ─► scripts d’extraction ─► data/annotations
                                      │
                                      ▼
                         consensus et contrôles
                                      │
                                      ▼
                              data/processed
```


## 10. État d’avancement

| Élément | État |
|---|---|
| exploration des quatre sources | terminée |
| phase 1, QA médicale générale | 2 622 exemples : 2 098 train, 262 validation, 262 test |
| consensus MediQAl | 700 cas retenus |
| consensus UltraMedical | 489 cas retenus |
| phase 2, triage bilingue | 2 378 exemples : 1 902 train, 238 validation, 238 test |
| équilibre linguistique phase 2 | 1 189 français + 1 189 anglais |
| corpus DPO disjoint | produit localement |
| audit automatique des 5 000 exemples SFT | terminé, aucun identifiant direct détecté |
| revue humaine d’échantillon | à réaliser |

Les deux phases possèdent leurs propres jeux. Le test reste à l’écart de l’entraînement et du choix des hyperparamètres.


## 11. Audit ultérieur avant les expériences finales

La préparation initiale produit 5 000 entrées. L’audit des expériences révèle ensuite des doublons de prompts et cinq paires déjà exposées au pilote de test. Je conserve les fichiers historiques et fige une version nettoyée avec `figer_partitions.py`.

| Corpus v3 | Train | Validation | Test | Total |
|---|---:|---:|---:|---:|
| QA | 2 098 | 262 | 262 | 2 622 |
| Triage | 1 888 | 238 | 226 | 2 352 |
| **SFT total** | **3 986** | **500** | **488** | **4 974** |

Le total inclut les partitions d’évaluation : les 4 974 entrées ne sont pas toutes entraînées. Je privilégie l’absence de fuite au maintien artificiel du compte de 5 000. Les deux langues d’un cas restent ensemble. Le DPO nettoyé contient 4 470 paires de train et 494 de validation ; un pilote de 256/64 paires sert à vérifier l’apprentissage.

Le [notebook 03](../evaluation/03_execution_colab.ipynb) expose les exclusions, le protocole et les mesures. L’audit automatique des identifiants ne remplace toujours pas la revue humaine ni la clarification des droits de réutilisation.
